# WavqWise: Dam Water Level Monitoring & Flood Early Warning
**Sense. Forecast. Alert.**

49 real dams across 13 countries. Filter by country, state, river. Forecast water levels. Detect flood risk.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VK-Ant/wavqwise/blob/main/demos/notebooks/wavqwise_dam_monitoring.ipynb)

**Real Data Sources (all free, no key):**
- India-WRIS: india-wris.nrsc.gov.in (5000+ Indian dams)
- USGS: waterservices.usgs.gov (1.5M+ US water sites)
- Global Dam Watch: globaldamwatch.org (7000+ worldwide)

**Author:** [VK-Ant](https://github.com/VK-Ant)

In [ ]:
!pip install wavqwise -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wavqwise import WavqPipeline
from wavqwise.anomaly.pipeline import AnomalyPipeline
from wavqwise.weather.dam_database import DamDB
from wavqwise.visualization.renderer import CLIPrinter
print('WavqWise loaded')

## 1. Explore Dam Database

In [ ]:
DamDB.summary()

## 2. Filter by Region
Change country/state to see your region.

In [ ]:
# Tamil Nadu dams
tn = DamDB.filter(country='India', state='Tamil Nadu')
print(f'Tamil Nadu: {len(tn)} dams\n')
for _, d in tn.iterrows():
    print(f'  {d["name"]} ({d["river"]}): capacity {d["capacity_ft"]}ft')

In [ ]:
# Try other regions
print('USA - California:'); print(DamDB.filter(country='USA', state='California')[['name','river','capacity_ft']].to_string(index=False))
print('\nAll Indian states:', DamDB.list_states('India'))
print('\nSearch "Cauvery":', DamDB.search('cauvery')[['name','state']].to_string(index=False))

## 3. Generate & Analyze Dam Data

In [ ]:
def generate_dam_data(dam, days=365):
    np.random.seed(hash(dam['name']) % 10000)
    dates = pd.date_range('2024-01-01', periods=days, freq='D')
    t = np.arange(days)/365
    seasonal = dam['capacity_ft']*0.2*np.sin(2*np.pi*(t-0.65))
    monsoon = dam['capacity_ft']*0.08*np.maximum(0, np.sin(2*np.pi*(t-0.7)))
    rain = np.random.exponential(3, days)*(1+2*np.maximum(0, np.sin(2*np.pi*(t-0.65))))
    inflow = np.convolve(rain, np.exp(-np.arange(10)/3), mode='same')
    noise = np.random.normal(0, dam['capacity_ft']*0.01, days)
    level = dam['capacity_ft']*0.6 + seasonal + monsoon + inflow*0.2 + noise
    level = np.clip(level, dam['capacity_ft']*0.1, dam['capacity_ft']*1.02)
    return pd.DataFrame({'date':dates, 'water_level_ft':np.round(level,1), 'rainfall_mm':np.round(np.maximum(0,rain),1)})

# Generate for selected region
REGION = DamDB.filter(country='India', state='Tamil Nadu')
dam_data = {}
for _, dam in REGION.iterrows():
    dam_data[dam['name']] = generate_dam_data(dam)
    latest = dam_data[dam['name']]['water_level_ft'].iloc[-1]
    pct = latest/dam['capacity_ft']*100
    status = 'CRITICAL' if pct>90 else 'HIGH' if pct>75 else 'NORMAL'
    print(f'{dam["name"]}: {latest:.0f}/{dam["capacity_ft"]}ft ({pct:.0f}%) [{status}]')

## 4. Forecast Water Levels

In [ ]:
forecasts = {}
for name, data in dam_data.items():
    p = WavqPipeline()
    p.load(data, target='water_level_ft', time='date')
    fc = p.forecast(horizon=30, model='ema')
    forecasts[name] = fc
    current = data['water_level_ft'].iloc[-1]
    pred = fc.forecast['water_level_ft'].iloc[-1]
    print(f'{name}: now={current:.0f}ft, 30d={pred:.0f}ft ({pred-current:+.0f}ft)')

## 5. Dam Network Visualization

In [ ]:
n = len(dam_data)
cols = min(3, n)
rows = (n+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
if n == 1: axes = np.array([[axes]])
elif rows == 1: axes = axes.reshape(1, -1)

for idx, (name, data) in enumerate(dam_data.items()):
    r, c = idx//cols, idx%cols
    ax = axes[r][c]
    dam_info = REGION[REGION['name']==name].iloc[0]
    cap = dam_info['capacity_ft']
    pct = data['water_level_ft'].iloc[-1]/cap*100
    color = '#dc2626' if pct>90 else '#f59e0b' if pct>75 else '#059669'
    status = 'CRITICAL' if pct>90 else 'HIGH' if pct>75 else 'NORMAL'
    ax.plot(data['date'], data['water_level_ft'], color='#2563eb', linewidth=1)
    fc = forecasts[name].forecast
    ax.plot(fc['date'], fc['water_level_ft'], '--', color='#dc2626', linewidth=2)
    ax.axhline(cap, color='#ef4444', linestyle=':', linewidth=1)
    ax.axhline(cap*0.9, color='#f59e0b', linestyle=':', linewidth=0.8)
    ax.set_title(f'{name}\n{data["water_level_ft"].iloc[-1]:.0f}/{cap}ft ({pct:.0f}%) [{status}]', fontsize=10, fontweight='bold', color=color)
    ax.grid(True, alpha=0.2)

for idx in range(n, rows*cols):
    axes[idx//cols][idx%cols].set_visible(False)

plt.suptitle(f'Dam Network: {REGION.iloc[0]["state"]}, {REGION.iloc[0]["country"]}', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Anomaly Detection (Flood Warning)

In [ ]:
for name, data in dam_data.items():
    det = AnomalyPipeline()
    det.load(data, target='water_level_ft', time='date')
    result = det.detect(method='zscore')
    n_anom = len(result.anomalies)
    if n_anom > 0:
        worst = result.anomalies.nlargest(1, 'anomaly_score').iloc[0]
        print(f'{name}: {n_anom} anomalies, worst score={worst["anomaly_score"]:.2f} ({worst["severity"]})')
    else:
        print(f'{name}: Clean - no anomalies')

## 7. Real-Time Streaming (Single Dam)

In [ ]:
# Pick first dam
dam_name = list(dam_data.keys())[0]
data = dam_data[dam_name]

p = WavqPipeline()
p.load(data.iloc[:350], target='water_level_ft', time='date')
p.forecast(horizon=7, model='ema')

alerts = []
stream = p.stream(model='ema', anomaly_method='zscore',
    on_anomaly=lambda e: alerts.append(e), anomaly_threshold=2.0, forecast_every=5)

# Simulate 15 days including a flood spike
dam_info = REGION[REGION['name']==dam_name].iloc[0]
for i in range(15):
    if 350+i < len(data):
        row = data.iloc[350+i]
        val = row['water_level_ft']
        if i == 10: val = dam_info['capacity_ft'] * 0.98  # Near overflow
        stream.push({'date': row['date'], 'water_level_ft': val})

print(stream.summary())
print(f'Flood alerts: {len(alerts)}')
for a in alerts:
    for _, r in a.anomalies.iterrows():
        print(f'  ALERT: {r["water_level_ft"]:.1f}ft, severity={r.get("severity","?")}, score={r.get("anomaly_score",0):.2f}')

## 8. Model Comparison

In [ ]:
p2 = WavqPipeline()
p2.load(data, target='water_level_ft', time='date')
comp = p2.compare_models(['moving_average', 'ema', 'naive', 'seasonal_naive'], horizon=14)
print(comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#059669' if i==0 else '#94a3b8' for i in range(len(comp))]
ax.barh(comp['model'], comp['MAE'], color=colors)
ax.set_title(f'Model Comparison: {dam_name}', fontweight='bold')
ax.set_xlabel('MAE (lower = better)'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

## 9. Try Another Region
Just change the filter:

In [ ]:
# Uncomment any:
# REGION = DamDB.filter(country='India', state='Kerala')
# REGION = DamDB.filter(country='USA', state='California')
# REGION = DamDB.filter(country='China')
# REGION = DamDB.filter(country='Brazil')
# REGION = DamDB.filter(river='Colorado')
# REGION = DamDB.filter(min_capacity=1000)  # All mega dams
print('Available countries:', DamDB.list_countries())

---
**WavqWise** - Sense. Forecast. Alert. | [GitHub](https://github.com/VK-Ant/wavqwise)

**Free data sources:** India-WRIS (india-wris.nrsc.gov.in) | USGS (waterservices.usgs.gov) | Global Dam Watch (globaldamwatch.org)